**Setup:** download `data/zip/subscription.zip` from the DA Launchpad Datasets page and unzip it in the same folder as this notebook, so that paths like `subscription/<table>.csv` work. Needs pandas (and matplotlib/seaborn for charts).

# Subscription funnel and cohort retention

This notebook measures the trial-to-paid conversion by platform and acquisition source. Then it builds a monthly cohort table: of the users who signed up in a month, how many were active 0, 1, 2 and 3 months later?

Put the CSV files for this dataset in a folder named `subscription` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import pandas as pd

users = pd.read_csv("subscription/users.csv").drop_duplicates(subset="user_id").copy()
subs = pd.read_csv("subscription/subscriptions.csv").drop_duplicates(subset="subscription_id").copy()
events = pd.read_csv("subscription/events.csv").drop_duplicates(subset="event_id").copy()
print("users:", len(users), "| subscriptions:", len(subs), "| events:", len(events))

def parse_date(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(s, format="%d-%m-%Y", errors="coerce")
    return iso.combine_first(dmy)

def parse_time(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    dmy = pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce")
    return iso.combine_first(dmy)

## Step 1: Trial to paid, by platform

A user is in the funnel if they started a trial. They count as paid if `paid_start_date` is filled in.

In [ ]:
m = subs.merge(users[["user_id", "platform"]], on="user_id", how="inner")
m["paid"] = m["paid_start_date"].notna()
by_platform = m.groupby("platform").agg(trial_users=("user_id", "size"), paid_users=("paid", "sum"))
by_platform["conversion_pct"] = (by_platform["paid_users"] / by_platform["trial_users"] * 100).round(1)
by_platform.sort_values("conversion_pct", ascending=False)

## Step 2: Trial to paid, by acquisition source

Source shows which campaigns bring users who pay. Check the row count, then compare.

In [ ]:
m = subs.merge(users[["user_id", "acquisition_source"]], on="user_id", how="inner")
print("joined rows:", len(m), "| subscriptions:", len(subs))
m["paid"] = m["paid_start_date"].notna()
by_source = m.groupby("acquisition_source").agg(trial_users=("user_id", "size"), paid_users=("paid", "sum"))
by_source["conversion_pct"] = (by_source["paid_users"] / by_source["trial_users"] * 100).round(1)
by_source.sort_values("conversion_pct", ascending=False)

## Step 3: Signup cohorts

Group users by the month they signed up. Parse the signup date first, because some rows use `DD-MM-YYYY`.

In [ ]:
users["signup_dt"] = parse_date(users["signup_date"])
users["cohort"] = users["signup_dt"].dt.to_period("M")
print("unparsed signup dates:", users["signup_dt"].isna().sum())
cohort_size = users.groupby("cohort").size().rename("cohort_users")
cohort_size.tail(6)

## Step 4: Activity by month

A user is active in a month if they have any event that month. Keep one row per user and month, so heavy users do not count twice.

In [ ]:
events["event_dt"] = parse_time(events["event_time"])
events["event_month"] = events["event_dt"].dt.to_period("M")
active = events[["user_id", "event_month"]].dropna().drop_duplicates()
active = active.merge(users[["user_id", "cohort"]], on="user_id", how="inner")
active["offset"] = (active["event_month"].dt.year - active["cohort"].dt.year) * 12 + (active["event_month"].dt.month - active["cohort"].dt.month)
print(active[["user_id", "cohort", "event_month", "offset"]].head())

## Step 5: Monthly cohort retention table

For each cohort, the share of its users who were active 0, 1, 2 and 3 months after signup. Month 0 is the signup month. A good table falls slowly from left to right.

In [ ]:
counts = active[active["offset"].between(0, 3)].groupby(["cohort", "offset"])["user_id"].nunique().unstack("offset")
retention = counts.div(cohort_size, axis=0).mul(100).round(1)
retention = retention.dropna(how="all")
retention.tail(12)

## Findings template

- Which platform and which source convert best? Give the rate and the number of trial users.
- Which cohort month falls fastest after month 0? Check that the cohort is not too small.
- Caveat: a user with no events in a month may still be a paying customer. Say that in your summary.